In [1]:
import pandas as pd

df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
# Drop customerID - it's just an identifier, not useful for prediction
df.drop('customerID', axis=1, inplace=True)

# TotalCharges is stored as text with some blank spaces - convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check how many rows became missing after conversion
print("Missing TotalCharges:", df['TotalCharges'].isnull().sum())

# Fill those missing values with the median
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

print("Missing values now:", df['TotalCharges'].isnull().sum())

Missing TotalCharges: 11
Missing values now: 11


C:\Users\sIMRAN\AppData\Local\Temp\ipykernel_10824\4117395930.py:11: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


In [3]:
from sklearn.preprocessing import LabelEncoder

# Convert target column: Churn (Yes/No) -> (1/0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Find all other text (categorical) columns
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print("Categorical columns to encode:", categorical_cols)

# Encode each categorical column into numbers
le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

df.head()

Categorical columns to encode: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


C:\Users\sIMRAN\AppData\Local\Temp\ipykernel_10824\1458224797.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include='object').columns.tolist()


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,29.85,0
1,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,1889.50,0
2,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,108.15,1
3,1,0,0,0,45,0,1,0,2,0,2,2,0,0,1,0,0,42.30,1840.75,0
4,0,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,151.65,1


In [4]:
from sklearn.model_selection import train_test_split

X = df.drop('Churn', axis=1)   # all columns except Churn
y = df['Churn']                # just the Churn column

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

Training set shape: (5634, 19)
Testing set shape: (1409, 19)


In [7]:
# Check which columns still have missing values
print(df.isnull().sum()[df.isnull().sum() > 0])

TotalCharges    11
dtype: int64


In [8]:
# Properly fill missing values (reassign instead of inplace)
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Confirm it's fixed
print(df.isnull().sum().sum())  # should print 0

0


In [10]:
print("NaN in X_train:", X_train.isnull().sum().sum())
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])

NaN in X_train: 8
TotalCharges    8
dtype: int64


In [11]:
# Fill remaining NaNs directly in the split data
X_train['TotalCharges'] = X_train['TotalCharges'].fillna(X_train['TotalCharges'].median())
X_test['TotalCharges'] = X_test['TotalCharges'].fillna(X_test['TotalCharges'].median())

print("NaN in X_train now:", X_train.isnull().sum().sum())
print("NaN in X_test now:", X_test.isnull().sum().sum())

NaN in X_train now: 0
NaN in X_test now: 0


In [12]:
from imblearn.over_sampling import SMOTE

print("Before SMOTE:", y_train.value_counts().to_dict())

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("After SMOTE:", y_train_balanced.value_counts().to_dict())

Before SMOTE: {0: 4139, 1: 1495}
After SMOTE: {0: 4139, 1: 4139}


In [13]:
import joblib

# Save the balanced training data and test data
joblib.dump(X_train_balanced, '../data/X_train_balanced.pkl')
joblib.dump(y_train_balanced, '../data/y_train_balanced.pkl')
joblib.dump(X_test, '../data/X_test.pkl')
joblib.dump(y_test, '../data/y_test.pkl')

# Also save the list of column names (we'll need this later for the app)
joblib.dump(X.columns.tolist(), '../data/feature_columns.pkl')

print("All files saved successfully!")

All files saved successfully!
